# Fitting an Observed Galaxy

This notebook runs the whole BayMorph pipeline on the configuration written by
`02A_Observed_Assessment_and_Config.ipynb`: image preparation (cutout,
background, noise and mask), the empirical PSF, the fit and its diagnostics.

Every cell can be run again. Accepted products are immutable, so a stage that
finds a completed product loads it and says so instead of rebuilding it, and
the fit resumes its Nautilus checkpoint. After an interruption, rerun the
notebook from the top: it continues where it stopped.

Changing a declaration is a different scientific problem. Go back to 02A and
choose a new `analysis_name`; products are never rewritten in place.

In [ ]:
import numpy as np
import yaml
from astropy.table import Table

from baymorph import BayMorphRunner
from baymorph.plotting import (
    corner_plot,
    plot_log_data_model_residual,
    plot_log_image,
)

## Configuration

`config_path` is the YAML written by 02A, resolved from the project root. It
declares every prior the fit samples. The sampler settings below add the
defaults BayMorph filled in.

In [ ]:
config_path = "configs/galaxy_r.yaml"

runner = BayMorphRunner(config_path)
object_name = runner.config["object"]["name"]

print(f"Configuration: {runner.config_path}")
print(f"Run directory: {runner.run_dir}")
print()
print(runner.config_path.read_text(encoding="utf-8"))
print("Resolved sampler settings:")
print(yaml.safe_dump(runner.config["fit"]["sampler"], sort_keys=False))

## Image preparation

`prepare_image()` cuts out the target, removes the sky, measures the noise
scale and builds the fitting mask. Masked pixels (neighbouring sources,
deblended fragments, artefacts) do not enter the likelihood. A fixed cutout
too small to contain the sky plateau around the galaxy is rejected here; the
remedy is a larger cutout, declared in 02A under a new `analysis_name`.

Look at the mask before fitting: large masked regions inside the galaxy, or
unmasked artefacts, change the scientific problem. The status line says
whether this stage built the products or reused accepted ones.

In [ ]:
image = runner.prepare_image()

height, width = image.prepared_data.shape

print(f"Cutout: {width} x {height} pixels ({image.cutout_mode})")
print(f"Pixels excluded by the fitting mask: {image.fitting_mask.mean():.1%}")
print(f"Noise scale: {np.median(image.noise):.4g}")

In [ ]:
figure, _ = plot_log_image(
    image.prepared_data,
    title=f"{object_name}: prepared cutout",
    figsize=(15, 15),
    show=True,
)

figure, _ = plot_log_image(
    np.where(image.fitting_mask, np.nan, image.prepared_data),
    title=f"{object_name}: pixels used by the likelihood",
    figsize=(15, 15),
    show=True,
)

### Sky gradient

Preparation measures the noise scale about a plane fitted to the sky and
reports the plane's gradient. The gradient is a diagnostic and never changes
the background model. A gradient significant at several sigma indicates that
`model: plane` is warranted, which is a new declaration in 02A.

A sky discontinuity, such as the edge of a coadd footprint, is modelled by
neither `constant` nor `plane`. A smaller fixed cutout can exclude it.

In [ ]:
gradient = image.sky_gradient

if gradient is None:
    print("No sky gradient was measured.")
else:
    for axis in ("x", "y"):
        value = gradient[f"gradient_{axis}"]
        error = gradient[f"gradient_{axis}_error"]
        significance = gradient[f"gradient_{axis}_significance"]

        print(
            f"gradient_{axis}: {value:.3g} +/- {error:.2g} per pixel "
            f"({significance:.1f} sigma)"
        )

## Empirical PSF

BayMorph selects isolated stars whose peak lies below `psf_max_peak` and
combines them into one PSF kernel. It rejects a kernel that is not finite,
non-negative, normalized and centred.

In [ ]:
runner.select_psf_candidates()
psf = runner.build_psf()

figure, _ = plot_log_image(
    psf,
    title=f"{object_name}: empirical PSF",
    figsize=(15, 15),
    show=True,
)

## Fit

`fit()` runs Nautilus with the declared priors and likelihood. This is the
slow cell: its cost grows with the number of pixels used by the likelihood.

In [ ]:
fit = runner.fit()

## Posterior

The median and `q16`-`q84` credible interval of each parameter. `x0` and `y0`
are in cutout pixel coordinates, and `background_level` is the residual
background after preparation, not the sky level.

In [ ]:
inference = fit.posterior

posterior_table = Table(
    rows=[
        {
            "parameter": name,
            **inference.summary(name),
        }
        for name in inference.parameter_names
    ]
)

posterior_table

## Residuals

Data, best-fitting model and residuals over the pixels used by the likelihood.
The normalized residual divides by the scale of the noise model at the best
sample, which adds the source Poisson variance to the sky noise when the
configuration declares an effective gain (AD-047). It is not whitened, so
neighbouring pixels keep the correlation of the sky. Structure in the residuals
shows where the morphology model or the observation model falls short.

In [ ]:
used = np.isfinite(fit.norm_residual)

residual_figure = plot_log_data_model_residual(
    np.where(used, fit.best_model_image + fit.residual, np.nan),
    fit.best_model_image,
    norm_residual=fit.norm_residual,
    title=object_name,
    show=True,
)

## Diagnostics

`diagnostics()` writes the corner plot, residual maps and radial profile as
PDF files in the run directory. The cell below also draws the corner plot
inline.

In [ ]:
runner.diagnostics()

print(f"Diagnostics written to: {runner.run_dir}")

In [ ]:
corner_figure, _, _ = corner_plot(
    inference.equal_weight_samples.T,
    parameter_labels=inference.parameter_names,
    title=object_name,
)

## Summary and limitations

The reduced chi-squared is the quadratic form of the noise model of AD-047
divided by the degrees of freedom. That model adds to the sky noise the
galaxy's own Poisson noise, through the effective gain the configuration
declares, and the correlation of the sky. A reduced chi-squared well above 1
means that the model does not describe the image within that noise: a misfit
of the morphology or of the background, which no noise model removes, or a
wrong noise model, for example a wrong gain. If the configuration declares
the source variance absent, the noise in the centre of a bright galaxy is
underestimated and the credible intervals are too narrow.
`prior_boundary_diagnostics` in the summary flags posteriors that press
against a declared prior bound.

In [ ]:
print(f"Backend: {inference.provenance.get('backend', 'unknown')}")
print(f"Reduced chi2: {fit.reduced_chi2:.4f} (ndof = {fit.ndof})")

if inference.logz is None:
    print("Evidence: unavailable (posterior-only backend)")
else:
    print(f"logZ: {inference.logz:.4f}")

fit.summary()